# Casino Game Revenue Prediction

## Data Science Module 1 Capstone

**Track:** B — Prediction Lab  
**Project Type:** Regression  
**Dataset:** Synthetic casino gaming data

In [5]:
PROJECT_CONFIG = {
    "track": "B",
    "title": "Casino Game Revenue Prediction",
    "stakeholder": "Casino gaming operations management",
    "decision_question": "Can machine, game, and location characteristics be used to predict monthly gaming revenue?",
    "dataset_source": "Synthetic casino gaming dataset created for educational capstone analysis",
    "row_grain": "One row represents one gaming machine at one location for one month",
    "target_column": "monthly_revenue",
    "id_columns": ["machine_id", "location_id"],
    "seed": 42,
    "test_size": 0.20,
}

SUCCESS_CRITERION = (
    "The predictive model should outperform a simple baseline model "
    "on unseen test data using regression metrics such as MAE, RMSE, and R-squared."
)

PROJECT_CONFIG

{'track': 'B',
 'title': 'Casino Game Revenue Prediction',
 'stakeholder': 'Casino gaming operations management',
 'decision_question': 'Can machine, game, and location characteristics be used to predict monthly gaming revenue?',
 'dataset_source': 'Synthetic casino gaming dataset created for educational capstone analysis',
 'row_grain': 'One row represents one gaming machine at one location for one month',
 'target_column': 'monthly_revenue',
 'id_columns': ['machine_id', 'location_id'],
 'seed': 42,
 'test_size': 0.2}

## 1. Problem and Project Design

### Project Question
Can machine, game, and location characteristics be used to predict monthly gaming revenue?

### Stakeholder
The intended stakeholder is casino gaming operations management. The results could help management better understand which machine, game, and location characteristics are associated with monthly revenue.

### Dataset Relevance
This project uses a synthetic casino gaming dataset created for educational purposes. The dataset contains information about gaming machines, locations, game types, operational activity, betting behavior, and monthly revenue.

### Unit of Analysis
Each row represents one gaming machine at one location for one month.

### Baseline
The project will first use a simple baseline prediction as a reference point. Machine-learning models will then be compared against the baseline to determine whether they provide better predictions.

### Useful Result
A useful result would be a model that predicts monthly revenue more accurately than the baseline on unseen test data. The model does not need to be perfect, but it should show measurable improvement using regression metrics.

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = PROJECT_CONFIG["seed"]

print("Libraries imported successfully.")
print("Random seed:", SEED)

Libraries imported successfully.
Random seed: 42


## 2. Dataset and Provenance

This project uses a synthetic casino gaming dataset created specifically for educational data science analysis. The dataset does not contain real casino, customer, or financial records.

The dataset contains monthly operational information for gaming machines across multiple fictional store locations. Each row represents one gaming machine at one location for one month.

The revised dataset includes both a unique `location_id` and a fictional `store_name` to make location-level analysis easier to interpret.

In [7]:
DATA_PATH = "casino_game_revenue_revised_with_store_names.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

Dataset loaded successfully.
Dataset shape: (2000, 16)


In [8]:
df.head()

,machine_id,location_id,store_name,month,game_type,location_type,distance_from_hq_miles,days_active,avg_bet,payout_rate,total_in,total_out,total_net,number_of_services,machine_swapouts,monthly_revenue
0,M0017,L040,Eagle Ridge Gaming,2025-06-01,Classic Reels,Gaming Hall,13.9,27,1.49,0.897,1877.52,1689.65,187.87,2,1,-400.18
1,M0140,L003,Lone Star Game Center,2026-04-01,A-Liner,Convenience Store,57.5,30,1.74,0.948,1200.00,1145.11,54.89,1,0,362.46
2,M0118,L018,Copper Creek Gaming,2025-01-01,Life of Luxury,Restaurant,18.0,21,2.24,0.915,3833.53,3388.67,444.86,1,0,-206.00
3,M0079,L021,Cedar Grove Gaming,2025-09-01,Classic Reels,Gaming Hall,50.0,26,1.20,0.858,2241.71,1876.70,365.01,0,0,87.87
4,M0078,L036,Greenfield Game Center,2026-03-01,Pot of Gold,Gaming Hall,32.5,23,2.45,NaN,7874.12,6880.70,993.42,0,0,951.83


In [9]:
df.columns.tolist()

['machine_id',
 'location_id',
 'store_name',
 'month',
 'game_type',
 'location_type',
 'distance_from_hq_miles',
 'days_active',
 'avg_bet',
 'payout_rate',
 'total_in',
 'total_out',
 'total_net',
 'number_of_services',
 'machine_swapouts',
 'monthly_revenue']

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   machine_id              2000 non-null   str    
 1   location_id             2000 non-null   str    
 2   store_name              2000 non-null   str    
 3   month                   2000 non-null   str    
 4   game_type               2000 non-null   str    
 5   location_type           2000 non-null   str    
 6   distance_from_hq_miles  1990 non-null   float64
 7   days_active             2000 non-null   int64  
 8   avg_bet                 1986 non-null   float64
 9   payout_rate             1988 non-null   float64
 10  total_in                2000 non-null   float64
 11  total_out               2000 non-null   float64
 12  total_net               2000 non-null   float64
 13  number_of_services      2000 non-null   int64  
 14  machine_swapouts        2000 non-null   int64  
 15

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   machine_id              2000 non-null   str    
 1   location_id             2000 non-null   str    
 2   store_name              2000 non-null   str    
 3   month                   2000 non-null   str    
 4   game_type               2000 non-null   str    
 5   location_type           2000 non-null   str    
 6   distance_from_hq_miles  1990 non-null   float64
 7   days_active             2000 non-null   int64  
 8   avg_bet                 1986 non-null   float64
 9   payout_rate             1988 non-null   float64
 10  total_in                2000 non-null   float64
 11  total_out               2000 non-null   float64
 12  total_net               2000 non-null   float64
 13  number_of_services      2000 non-null   int64  
 14  machine_swapouts        2000 non-null   int64  
 15

In [12]:
df.isnull().sum()

machine_id                 0
location_id                0
store_name                 0
month                      0
game_type                  0
location_type              0
distance_from_hq_miles    10
days_active                0
avg_bet                   14
payout_rate               12
total_in                   0
total_out                  0
total_net                  0
number_of_services         0
machine_swapouts           0
monthly_revenue            0
dtype: int64

In [13]:
df.duplicated().sum()

np.int64(0)

In [14]:
df.describe()

,distance_from_hq_miles,days_active,avg_bet,payout_rate,total_in,total_out,total_net,number_of_services,machine_swapouts,monthly_revenue
count,1990.000000,2000.000000,1986.000000,1988.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,56.913518,25.524000,1.648459,0.910109,2463.218345,2243.627275,219.591070,0.758000,0.138500,46.593690
std,37.993645,3.462574,0.544535,0.024683,1481.670733,1354.796362,151.865676,0.907109,0.362469,593.981576
min,2.000000,20.000000,0.250000,0.824000,1200.000000,1000.930000,20.810000,0.000000,0.000000,-900.000000
25%,28.225000,22.000000,1.280000,0.893000,1341.440000,1226.165000,120.127500,0.000000,0.000000,-398.417500
50%,48.700000,26.000000,1.640000,0.910000,2040.800000,1841.435000,173.840000,1.000000,0.000000,40.655000
75%,77.175000,29.000000,2.010000,0.927000,2994.225000,2717.267500,270.295000,1.000000,0.000000,461.227500
max,180.000000,31.000000,3.360000,0.970000,13338.130000,12075.380000,1487.660000,6.000000,2.000000,2377.520000


In [15]:
df["monthly_revenue"].describe()

count    2000.000000
mean       46.593690
std       593.981576
min      -900.000000
25%      -398.417500
50%        40.655000
75%       461.227500
max      2377.520000
Name: monthly_revenue, dtype: float64

In [18]:
df.duplicated(subset=["machine_id", "location_id", "month"]).sum()

np.int64(9)